# CEI Internship — Week 3: SQL Advanced Analysis
## Superstore Dataset: Subqueries, CTEs & Window Functions

**Objective:** Analyze sales data using SQL by applying Subqueries, CTEs, and Window Functions to solve business queries.

---
| Concept | Queries |
|---|---|
| Subqueries | Above-avg sales, highest order per customer |
| CTEs | Total sales per customer, above-avg customers |
| Window Functions | RANK, DENSE_RANK, ROW_NUMBER + PARTITION BY |
| Combined | JOIN + CTE + Window in one final query |
| Mini Project | Top 5, Bottom 5, Single-order, Above-avg, Max order |

## Setup

In [1]:
import sqlite3
import pandas as pd

# Helper to display query results nicely
def run_query(conn, sql, title=''):
    if title:
        print(f'\n{title}')
    df = pd.read_sql_query(sql, conn)
    display(df)
    return df

print('Libraries loaded successfully')

Libraries loaded successfully


---
## Step 1 — Data Setup
### 1.1  Load CSV into `superstore_raw`

In [2]:
# Create in-memory SQLite database
conn = sqlite3.connect(':memory:')

# Load the Superstore CSV
# Download from: https://www.kaggle.com/datasets/vivek468/superstore-dataset-final
df = pd.read_csv('superstore.csv', encoding='latin1')
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('-', '_', regex=False)
)

df.to_sql('superstore_raw', conn, if_exists='replace', index=False)
print(f'Loaded: {len(df):,} rows  |  {len(df.columns)} columns')
df.head(3)

Loaded: 9,994 rows  |  21 columns


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2,0.0,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3,0.0,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2,0.0,6.8714


### 1.2  Create Normalised Tables

In [3]:
conn.executescript("""
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;

-- Customers dimension
CREATE TABLE customers AS
SELECT DISTINCT
    customer_id, customer_name, segment,
    country, city, state, postal_code, region
FROM superstore_raw;

-- Products dimension
CREATE TABLE products AS
SELECT DISTINCT
    product_id, product_name, category, sub_category
FROM superstore_raw;

-- Orders fact table
CREATE TABLE orders AS
SELECT
    row_id, order_id, order_date, ship_date, ship_mode,
    customer_id, product_id,
    sales, quantity, discount, profit
FROM superstore_raw;
""")

for tbl in ['superstore_raw', 'customers', 'products', 'orders']:
    n = conn.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl:<20} {n:>6,} rows')

  superstore_raw        9,994 rows
  customers             4,910 rows
  products              1,894 rows
  orders                9,994 rows


---
## Step 2 — Required Queries
### Q1 — Orders with Sales > Average Sales  *(Subquery)*

In [4]:
run_query(conn, """
SELECT
    o.order_id,
    c.customer_name,
    p.category,
    ROUND(o.sales, 2)                          AS sales,
    ROUND((SELECT AVG(sales) FROM orders), 2)  AS avg_sales
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
JOIN products  p ON o.product_id  = p.product_id
WHERE o.sales > (SELECT AVG(sales) FROM orders)
ORDER BY o.sales DESC
""", 'Q1: Above-Average Sales Orders')


Q1: Above-Average Sales Orders


,order_id,customer_name,category,sales,avg_sales
0,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
1,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
2,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
3,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
4,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
...,...,...,...,...,...
17750,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86
17751,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86
17752,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86
17753,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86


,order_id,customer_name,category,sales,avg_sales
0,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
1,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
2,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
3,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
4,CA-2014-145317,Sean Miller,Technology,22638.48,229.86
...,...,...,...,...,...
17750,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86
17751,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86
17752,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86
17753,CA-2014-134103,Mike Vittorini,Office Supplies,229.94,229.86


### Q2 — Highest Sales Order per Customer  *(Subquery)*

In [5]:
run_query(conn, """
SELECT
    c.customer_name,
    o.order_id,
    ROUND(o.sales, 2) AS max_order_sales
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE o.sales = (
    SELECT MAX(o2.sales)
    FROM orders o2
    WHERE o2.customer_id = o.customer_id
)
ORDER BY o.sales DESC
""", 'Q2: Highest Sales Order per Customer')


Q2: Highest Sales Order per Customer


,customer_name,order_id,max_order_sales
0,Sean Miller,CA-2014-145317,22638.48
1,Sean Miller,CA-2014-145317,22638.48
2,Sean Miller,CA-2014-145317,22638.48
3,Sean Miller,CA-2014-145317,22638.48
4,Sean Miller,CA-2014-145317,22638.48
...,...,...,...
4919,Roy Skaria,CA-2016-107475,9.65
4920,Roy Skaria,CA-2016-107475,9.65
4921,Lela Donovan,CA-2016-152331,5.30
4922,Thais Sissman,US-2017-168690,2.81


,customer_name,order_id,max_order_sales
0,Sean Miller,CA-2014-145317,22638.48
1,Sean Miller,CA-2014-145317,22638.48
2,Sean Miller,CA-2014-145317,22638.48
3,Sean Miller,CA-2014-145317,22638.48
4,Sean Miller,CA-2014-145317,22638.48
...,...,...,...
4919,Roy Skaria,CA-2016-107475,9.65
4920,Roy Skaria,CA-2016-107475,9.65
4921,Lela Donovan,CA-2016-152331,5.30
4922,Thais Sissman,US-2017-168690,2.81


### Q3 — Total Sales per Customer  *(CTE)*

In [6]:
run_query(conn, """
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT * FROM customer_sales ORDER BY total_sales DESC
""", 'Q3: Total Sales per Customer (CTE)')


Q3: Total Sales per Customer (CTE)


,customer_id,customer_name,total_sales
0,KL-16645,Ken Lonsdale,155927.52
1,SE-20110,Sanjit Engle,134303.82
2,CL-12565,Clay Ludtke,130566.55
3,AB-10105,Adrian Barton,130262.14
4,SC-20095,Sanjit Chand,127281.01
...,...,...,...
788,RS-19870,Roy Skaria,44.66
789,MG-18205,Mitch Gastineau,16.74
790,CJ-11875,Carl Jackson,16.52
791,TS-21085,Thais Sissman,9.67


,customer_id,customer_name,total_sales
0,KL-16645,Ken Lonsdale,155927.52
1,SE-20110,Sanjit Engle,134303.82
2,CL-12565,Clay Ludtke,130566.55
3,AB-10105,Adrian Barton,130262.14
4,SC-20095,Sanjit Chand,127281.01
...,...,...,...
788,RS-19870,Roy Skaria,44.66
789,MG-18205,Mitch Gastineau,16.74
790,CJ-11875,Carl Jackson,16.52
791,TS-21085,Thais Sissman,9.67


### Q4 — Customers with Above-Average Total Sales  *(CTE + Subquery)*

In [7]:
run_query(conn, """
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT customer_name, total_sales
FROM customer_sales
WHERE total_sales > (SELECT AVG(total_sales) FROM customer_sales)
ORDER BY total_sales DESC
""", 'Q4: Above-Average Customers (CTE + Subquery)')


Q4: Above-Average Customers (CTE + Subquery)


,customer_name,total_sales
0,Ken Lonsdale,155927.52
1,Sanjit Engle,134303.82
2,Clay Ludtke,130566.55
3,Adrian Barton,130262.14
4,Sanjit Chand,127281.01
...,...,...
275,Alan Schoenberger,21303.92
276,Bill Eplett,21023.38
277,Dean Braden,20993.19
278,Frank Atkinson,20888.38


,customer_name,total_sales
0,Ken Lonsdale,155927.52
1,Sanjit Engle,134303.82
2,Clay Ludtke,130566.55
3,Adrian Barton,130262.14
4,Sanjit Chand,127281.01
...,...,...
275,Alan Schoenberger,21303.92
276,Bill Eplett,21023.38
277,Dean Braden,20993.19
278,Frank Atkinson,20888.38


### Q5 — Rank Customers by Total Sales  *(Window Function)*

In [8]:
run_query(conn, """
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
)
SELECT
    customer_name,
    total_sales,
    RANK()       OVER (ORDER BY total_sales DESC) AS sales_rank,
    DENSE_RANK() OVER (ORDER BY total_sales DESC) AS dense_rank
FROM customer_sales
ORDER BY sales_rank
""", 'Q5: Customer Ranking (RANK & DENSE_RANK)')


Q5: Customer Ranking (RANK & DENSE_RANK)


,customer_name,total_sales,sales_rank,dense_rank
0,Ken Lonsdale,155927.52,1,1
1,Sanjit Engle,134303.82,2,2
2,Clay Ludtke,130566.55,3,3
3,Adrian Barton,130262.14,4,4
4,Sanjit Chand,127281.01,5,5
...,...,...,...,...
788,Roy Skaria,44.66,789,789
789,Mitch Gastineau,16.74,790,790
790,Carl Jackson,16.52,791,791
791,Thais Sissman,9.67,792,792


,customer_name,total_sales,sales_rank,dense_rank
0,Ken Lonsdale,155927.52,1,1
1,Sanjit Engle,134303.82,2,2
2,Clay Ludtke,130566.55,3,3
3,Adrian Barton,130262.14,4,4
4,Sanjit Chand,127281.01,5,5
...,...,...,...,...
788,Roy Skaria,44.66,789,789
789,Mitch Gastineau,16.74,790,790
790,Carl Jackson,16.52,791,791
791,Thais Sissman,9.67,792,792


### Q6 — Order Sequence per Customer  *(ROW_NUMBER + PARTITION BY)*

In [9]:
run_query(conn, """
SELECT
    c.customer_name,
    o.order_id,
    o.order_date,
    ROUND(o.sales, 2) AS sales,
    ROW_NUMBER() OVER (
        PARTITION BY o.customer_id
        ORDER BY o.order_date ASC
    ) AS order_sequence
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
ORDER BY c.customer_name, order_sequence
LIMIT 30
""", 'Q6: ROW_NUMBER per Customer (PARTITION BY)')


Q6: ROW_NUMBER per Customer (PARTITION BY)


,customer_name,order_id,order_date,sales,order_sequence
0,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,1
1,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,2
2,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,3
3,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,4
4,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,5
5,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,6
6,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,7
7,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,8
8,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,9
9,Aaron Bergman,CA-2014-156587,3/7/2014,48.71,10


,customer_name,order_id,order_date,sales,order_sequence
0,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,1
1,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,2
2,Aaron Bergman,CA-2016-140935,11/10/2016,221.98,3
3,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,4
4,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,5
5,Aaron Bergman,CA-2016-140935,11/10/2016,341.96,6
6,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,7
7,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,8
8,Aaron Bergman,CA-2014-152905,2/18/2014,12.62,9
9,Aaron Bergman,CA-2014-156587,3/7/2014,48.71,10


### Q7 — Top 3 Customers  *(Window Function)*

In [10]:
run_query(conn, """
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name
),
ranked AS (
    SELECT customer_name, total_sales,
           RANK() OVER (ORDER BY total_sales DESC) AS rnk
    FROM customer_sales
)
SELECT customer_name, total_sales, rnk
FROM ranked
WHERE rnk <= 3
""", 'Q7: Top 3 Customers')


Q7: Top 3 Customers


,customer_name,total_sales,rnk
0,Ken Lonsdale,155927.52,1
1,Sanjit Engle,134303.82,2
2,Clay Ludtke,130566.55,3


,customer_name,total_sales,rnk
0,Ken Lonsdale,155927.52,1
1,Sanjit Engle,134303.82,2
2,Clay Ludtke,130566.55,3


---
## Step 3 — Final Combined Query
**Customer Name | Total Sales | Rank** — using JOIN + CTE + Window Function

In [11]:
run_query(conn, """
WITH customer_sales AS (
    SELECT
        o.customer_id,
        c.customer_name,
        c.segment,
        c.region,
        ROUND(SUM(o.sales), 2)    AS total_sales,
        COUNT(DISTINCT o.order_id) AS total_orders,
        ROUND(AVG(o.sales), 2)    AS avg_order_value
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name, c.segment, c.region
),
ranked_customers AS (
    SELECT
        customer_name, segment, region,
        total_sales, total_orders, avg_order_value,
        RANK() OVER (ORDER BY total_sales DESC)                      AS overall_rank,
        RANK() OVER (PARTITION BY segment ORDER BY total_sales DESC) AS segment_rank
    FROM customer_sales
)
SELECT
    overall_rank  AS rank,
    customer_name,
    segment,
    region,
    total_sales,
    total_orders,
    avg_order_value,
    segment_rank
FROM ranked_customers
ORDER BY overall_rank
""", 'Final Combined Query: Customer Name | Total Sales | Rank')


Final Combined Query: Customer Name | Total Sales | Rank


,rank,customer_name,segment,region,total_sales,total_orders,avg_order_value,segment_rank
0,1,Sanjit Engle,Consumer,West,85466.07,11,642.60,1
1,2,Seth Vernon,Consumer,East,80296.65,10,358.47,2
2,3,Adrian Barton,Consumer,Central,72367.85,10,723.68,3
3,4,Ken Lonsdale,Consumer,West,70876.15,12,488.80,4
4,5,Sanjit Chand,Consumer,West,70711.67,9,642.83,5
...,...,...,...,...,...,...,...,...
2496,2497,Mitch Gastineau,Corporate,South,16.74,1,8.37,740
2497,2498,Carl Jackson,Corporate,East,16.52,1,16.52,741
2498,2499,Lela Donovan,Corporate,Central,5.30,1,5.30,742
2499,2500,Thais Sissman,Consumer,South,4.83,2,2.42,1305


,rank,customer_name,segment,region,total_sales,total_orders,avg_order_value,segment_rank
0,1,Sanjit Engle,Consumer,West,85466.07,11,642.60,1
1,2,Seth Vernon,Consumer,East,80296.65,10,358.47,2
2,3,Adrian Barton,Consumer,Central,72367.85,10,723.68,3
3,4,Ken Lonsdale,Consumer,West,70876.15,12,488.80,4
4,5,Sanjit Chand,Consumer,West,70711.67,9,642.83,5
...,...,...,...,...,...,...,...,...
2496,2497,Mitch Gastineau,Corporate,South,16.74,1,8.37,740
2497,2498,Carl Jackson,Corporate,East,16.52,1,16.52,741
2498,2499,Lela Donovan,Corporate,Central,5.30,1,5.30,742
2499,2500,Thais Sissman,Consumer,South,4.83,2,2.42,1305


---
## Mini Project — Customer Sales Insights
### MI-1: Top 5 Customers

In [12]:
run_query(conn, """
WITH cs AS (
    SELECT c.customer_name, c.segment, c.region,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name, c.segment, c.region
)
SELECT RANK() OVER (ORDER BY total_sales DESC) AS rank,
       customer_name, segment, region, total_sales
FROM cs ORDER BY total_sales DESC LIMIT 5
""", 'MI-1: Top 5 Customers')


MI-1: Top 5 Customers


,rank,customer_name,segment,region,total_sales
0,1,Sanjit Engle,Consumer,West,85466.07
1,2,Seth Vernon,Consumer,East,80296.65
2,3,Adrian Barton,Consumer,Central,72367.85
3,4,Ken Lonsdale,Consumer,West,70876.15
4,5,Sanjit Chand,Consumer,West,70711.67


,rank,customer_name,segment,region,total_sales
0,1,Sanjit Engle,Consumer,West,85466.07
1,2,Seth Vernon,Consumer,East,80296.65
2,3,Adrian Barton,Consumer,Central,72367.85
3,4,Ken Lonsdale,Consumer,West,70876.15
4,5,Sanjit Chand,Consumer,West,70711.67


### MI-2: Bottom 5 Customers

In [13]:
run_query(conn, """
WITH cs AS (
    SELECT c.customer_name, c.segment, c.region,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY o.customer_id, c.customer_name, c.segment, c.region
)
SELECT RANK() OVER (ORDER BY total_sales ASC) AS rank,
       customer_name, segment, region, total_sales
FROM cs ORDER BY total_sales ASC LIMIT 5
""", 'MI-2: Bottom 5 Customers')


MI-2: Bottom 5 Customers


,rank,customer_name,segment,region,total_sales
0,1,Thais Sissman,Consumer,South,4.83
1,1,Thais Sissman,Consumer,West,4.83
2,3,Lela Donovan,Corporate,Central,5.30
3,4,Carl Jackson,Corporate,East,16.52
4,5,Mitch Gastineau,Corporate,South,16.74


,rank,customer_name,segment,region,total_sales
0,1,Thais Sissman,Consumer,South,4.83
1,1,Thais Sissman,Consumer,West,4.83
2,3,Lela Donovan,Corporate,Central,5.30
3,4,Carl Jackson,Corporate,East,16.52
4,5,Mitch Gastineau,Corporate,South,16.74


### MI-3: Customers with Only One Order

In [14]:
run_query(conn, """
SELECT c.customer_name, c.segment,
       COUNT(DISTINCT o.order_id) AS order_count
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY o.customer_id, c.customer_name, c.segment
HAVING COUNT(DISTINCT o.order_id) = 1
ORDER BY c.customer_name
""", 'MI-3: Single-Order Customers')


MI-3: Single-Order Customers


,customer_name,segment,order_count
0,Anemone Ratner,Consumer,1
1,Anthony O'Donnell,Corporate,1
2,Carl Jackson,Corporate,1
3,Jenna Caffey,Consumer,1
4,Jocasta Rupert,Consumer,1
5,Lela Donovan,Corporate,1
6,Mitch Gastineau,Corporate,1
7,Patricia Hirasaki,Home Office,1
8,Ricardo Emerson,Consumer,1
9,Roland Murray,Consumer,1


,customer_name,segment,order_count
0,Anemone Ratner,Consumer,1
1,Anthony O'Donnell,Corporate,1
2,Carl Jackson,Corporate,1
3,Jenna Caffey,Consumer,1
4,Jocasta Rupert,Consumer,1
5,Lela Donovan,Corporate,1
6,Mitch Gastineau,Corporate,1
7,Patricia Hirasaki,Home Office,1
8,Ricardo Emerson,Consumer,1
9,Roland Murray,Consumer,1


### MI-4: Customers with Above-Average Total Sales

In [15]:
run_query(conn, """
WITH cs AS (
    SELECT c.customer_id, c.customer_name, c.segment,
           ROUND(SUM(o.sales), 2) AS total_sales
    FROM orders o JOIN customers c ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, c.customer_name, c.segment
),
avg_s AS (SELECT ROUND(AVG(total_sales), 2) AS avg_total FROM cs)
SELECT cs.customer_name, cs.segment, cs.total_sales,
       av.avg_total AS overall_avg,
       ROUND(cs.total_sales - av.avg_total, 2) AS above_avg_by
FROM cs CROSS JOIN avg_s av
WHERE cs.total_sales > av.avg_total
ORDER BY cs.total_sales DESC
""", 'MI-4: Above-Average Sales Customers')


MI-4: Above-Average Sales Customers


,customer_name,segment,total_sales,overall_avg,above_avg_by
0,Ken Lonsdale,Consumer,155927.52,20667.9,135259.62
1,Sanjit Engle,Consumer,134303.82,20667.9,113635.92
2,Clay Ludtke,Consumer,130566.55,20667.9,109898.65
3,Adrian Barton,Consumer,130262.14,20667.9,109594.24
4,Sanjit Chand,Consumer,127281.01,20667.9,106613.11
...,...,...,...,...,...
275,Alan Schoenberger,Corporate,21303.92,20667.9,636.02
276,Bill Eplett,Home Office,21023.38,20667.9,355.48
277,Dean Braden,Consumer,20993.19,20667.9,325.29
278,Frank Atkinson,Corporate,20888.38,20667.9,220.48


,customer_name,segment,total_sales,overall_avg,above_avg_by
0,Ken Lonsdale,Consumer,155927.52,20667.9,135259.62
1,Sanjit Engle,Consumer,134303.82,20667.9,113635.92
2,Clay Ludtke,Consumer,130566.55,20667.9,109898.65
3,Adrian Barton,Consumer,130262.14,20667.9,109594.24
4,Sanjit Chand,Consumer,127281.01,20667.9,106613.11
...,...,...,...,...,...
275,Alan Schoenberger,Corporate,21303.92,20667.9,636.02
276,Bill Eplett,Home Office,21023.38,20667.9,355.48
277,Dean Braden,Consumer,20993.19,20667.9,325.29
278,Frank Atkinson,Corporate,20888.38,20667.9,220.48


### MI-5: Highest Order Value per Customer

In [16]:
run_query(conn, """
WITH order_totals AS (
    SELECT o.customer_id, o.order_id, o.order_date,
           ROUND(SUM(o.sales), 2) AS order_total
    FROM orders o
    GROUP BY o.customer_id, o.order_id, o.order_date
),
ranked_orders AS (
    SELECT *,
           RANK() OVER (PARTITION BY customer_id ORDER BY order_total DESC) AS rnk
    FROM order_totals
)
SELECT c.customer_name, c.segment,
       r.order_id, r.order_date, r.order_total AS highest_order_value
FROM ranked_orders r
JOIN customers c ON r.customer_id = c.customer_id
WHERE r.rnk = 1
ORDER BY r.order_total DESC
""", 'MI-5: Highest Order Value per Customer')


MI-5: Highest Order Value per Customer


,customer_name,segment,order_id,order_date,highest_order_value
0,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
1,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
2,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
3,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
4,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
...,...,...,...,...,...
4905,Roy Skaria,Home Office,CA-2017-142776,12/11/2017,12.68
4906,Roy Skaria,Home Office,CA-2017-142776,12/11/2017,12.68
4907,Lela Donovan,Corporate,CA-2016-152331,6/26/2016,5.30
4908,Thais Sissman,Consumer,US-2017-168690,1/7/2017,2.81


,customer_name,segment,order_id,order_date,highest_order_value
0,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
1,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
2,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
3,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
4,Sean Miller,Home Office,CA-2014-145317,3/18/2014,23661.23
...,...,...,...,...,...
4905,Roy Skaria,Home Office,CA-2017-142776,12/11/2017,12.68
4906,Roy Skaria,Home Office,CA-2017-142776,12/11/2017,12.68
4907,Lela Donovan,Corporate,CA-2016-152331,6/26/2016,5.30
4908,Thais Sissman,Consumer,US-2017-168690,1/7/2017,2.81


---
## Business Insights

| # | Insight | Action |
|---|---------|--------|
| 1 | **Top 3 customers** generate a disproportionate share of total revenue | Launch VIP loyalty program |
| 2 | **Single-order customers** are churn risks | Re-engagement email campaigns |
| 3 | **High-discount orders (>40%)** frequently show negative profit | Review discount policy |
| 4 | **Corporate segment** places larger individual orders | Assign dedicated account managers |
| 5 | **West & East regions** lead in volume; Central & South lag | Target marketing in under-served regions |
| 6 | **Technology category** has highest per-unit sales | Prioritise inventory & promotions |

In [17]:
conn.close()
print('Analysis complete. Connection closed.')

Analysis complete. Connection closed.
